<a href="https://colab.research.google.com/github/khalidkhankakar/Hands-on-Machine-Learning/blob/master/chapter_10_ANN/neural_network_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Linear Reqression with Pytorch

In [1]:
import torch
from sklearn.datasets import fetch_california_housing
X, y = fetch_california_housing(return_X_y=True)
X.shape, y.shape


((20640, 8), (20640,))

In [2]:
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42)
X_train, X_dev, y_train, y_dev = train_test_split(X_train_full, y_train_full, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_dev.shape, y_dev.shape

((11610, 8), (11610,), (5160, 8), (5160,), (3870, 8), (3870,))

In [3]:
X_train = torch.FloatTensor(X_train)
X_dev = torch.FloatTensor(X_dev)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_test = (X_test- means) / stds
X_dev = (X_dev - means) / stds

In [4]:
y_train= torch.FloatTensor(y_train.reshape(-1, 1))
y_dev= torch.FloatTensor(y_dev.reshape(-1, 1))
y_test= torch.FloatTensor(y_test.reshape(-1, 1))

In [5]:
# Our parameter for network
n_features = X_train.shape[1]
weights = torch.randn((n_features, 1), requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

In [6]:
# let's train the model
learning_rate = 0.01
n_epochs = 20

for epoch in range(n_epochs):
  y_pred = X_train @ weights + bias
  loss = ((y_pred - y_train)**2).mean()
  loss.backward()
  with torch.no_grad():
    bias -= learning_rate * bias.grad
    weights -= learning_rate * weights.grad
    bias.grad.zero_()
    weights.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss {loss.item()}")

Epoch 1/20, Loss 7.312067985534668
Epoch 2/20, Loss 7.034704685211182
Epoch 3/20, Loss 6.7690300941467285
Epoch 4/20, Loss 6.514526844024658
Epoch 5/20, Loss 6.27070426940918
Epoch 6/20, Loss 6.037092685699463
Epoch 7/20, Loss 5.81324577331543
Epoch 8/20, Loss 5.598736763000488
Epoch 9/20, Loss 5.393160343170166
Epoch 10/20, Loss 5.196130275726318
Epoch 11/20, Loss 5.007274627685547
Epoch 12/20, Loss 4.8262434005737305
Epoch 13/20, Loss 4.652699947357178
Epoch 14/20, Loss 4.48632287979126
Epoch 15/20, Loss 4.326805591583252
Epoch 16/20, Loss 4.17385721206665
Epoch 17/20, Loss 4.027195930480957
Epoch 18/20, Loss 3.8865573406219482
Epoch 19/20, Loss 3.751685380935669
Epoch 20/20, Loss 3.6223366260528564


In [7]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = X_new @ weights + bias

In [8]:
y_pred

tensor([[-0.2628],
        [ 0.6953],
        [ 1.2879]])

Linear Regression with torch high level API

In [9]:
import torch.nn as nn
model = nn.Linear(in_features= n_features, out_features = 1)

In [10]:
# parameters
model.bias, model.weight

(Parameter containing:
 tensor([-0.1912], requires_grad=True),
 Parameter containing:
 tensor([[ 0.0830, -0.0824,  0.0409,  0.2875,  0.3034, -0.1477, -0.0691, -0.3288]],
        requires_grad=True))

In [11]:
for param in model.parameters():
  print(param)
for named_param in model.named_parameters():
  print(named_param[0], named_param[1])

Parameter containing:
tensor([[ 0.0830, -0.0824,  0.0409,  0.2875,  0.3034, -0.1477, -0.0691, -0.3288]],
       requires_grad=True)
Parameter containing:
tensor([-0.1912], requires_grad=True)
weight Parameter containing:
tensor([[ 0.0830, -0.0824,  0.0409,  0.2875,  0.3034, -0.1477, -0.0691, -0.3288]],
       requires_grad=True)
bias Parameter containing:
tensor([-0.1912], requires_grad=True)


In [12]:
# here model is just normal.
# so model is not train yet. it will make terriable predications
# Note: when call the this torch will internall call forward() method Which is just as see in previous example. X @ self.weight + self.bias
model(X_train[:3])

tensor([[0.3414],
        [0.2119],
        [0.5943]], grad_fn=<AddmmBackward0>)

In [13]:
# optimizer: In short it updates the learnable parameters values like weights and bias. You can choose different kinds algorithm to perform backpropagation
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [14]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'{epoch}/{n_epochs}, loss: {loss.item()}')

In [15]:
train_bgd(model, optimizer, mse, X_train, y_train, 20)

0/20, loss: 6.621084213256836
1/20, loss: 6.378017425537109
2/20, loss: 6.144887447357178
3/20, loss: 5.921275615692139
4/20, loss: 5.706783771514893
5/20, loss: 5.501030445098877
6/20, loss: 5.303651809692383
7/20, loss: 5.114297866821289
8/20, loss: 4.932635307312012
9/20, loss: 4.758345127105713
10/20, loss: 4.591121196746826
11/20, loss: 4.4306721687316895
12/20, loss: 4.276716709136963
13/20, loss: 4.128988265991211
14/20, loss: 3.987229585647583
15/20, loss: 3.8511948585510254
16/20, loss: 3.720649003982544
17/20, loss: 3.595367670059204
18/20, loss: 3.4751338958740234
19/20, loss: 3.3597424030303955


Implement MLP with low level API

In [39]:
# input layer weights and bias
w1 = torch.randn((n_features, 64), requires_grad=True)
b1 = torch.zeros(64, requires_grad=True)

# second layer weights and bias
w2 = torch.randn((64, 32), requires_grad=True)
b2 = torch.zeros(32, requires_grad=True)

# Output layer weights and bias
w3 = torch.randn((32, 1), requires_grad=True)
b3 = torch.zeros(1, requires_grad=True)

parameters = [w1, b1, w2, b2, w3, b3]
print(sum(p.nelement() for p in parameters))

2689


In [49]:
n_epochs = 50
for epoch in range(n_epochs):
  # forward pass
  x = X_train @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3
  loss = ((y_pred - y_train)**2).mean()


  # backward pass
  for p in parameters:
    p.grad= None
  loss.backward()

  # update
  lr = 0.001
  with torch.no_grad():
    for p in parameters:
      p.data -= lr * p.grad

  print(f'{epoch}/{n_epochs}, loss: {loss.item():.4f}')


0/20, loss: 2.5711
1/20, loss: 2.5630
2/20, loss: 2.5550
3/20, loss: 2.5470
4/20, loss: 2.5390
5/20, loss: 2.5312
6/20, loss: 2.5234
7/20, loss: 2.5157
8/20, loss: 2.5080
9/20, loss: 2.5004
10/20, loss: 2.4929
11/20, loss: 2.4854
12/20, loss: 2.4779
13/20, loss: 2.4706
14/20, loss: 2.4633
15/20, loss: 2.4560
16/20, loss: 2.4489
17/20, loss: 2.4417
18/20, loss: 2.4346
19/20, loss: 2.4276


In [50]:
# predications
with torch.no_grad():
  x = X_dev[:10] @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3

print(y_pred)


tensor([[ 1.3461],
        [ 1.3415],
        [ 3.0294],
        [ 0.8859],
        [ 3.8230],
        [ 0.2729],
        [ 2.1899],
        [ 1.5703],
        [ 1.7982],
        [-0.8696]])
